# INNER È 5

In [ ]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from imblearn import FunctionSampler
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

DATASET = Path("Datasets/Processed/Dataset Dupilumab Baseline.xlsx - Anagrafica.csv")


def load_dataset():
    for root in (Path.cwd(), *Path.cwd().parents):
        if (root / DATASET).is_file():
            return pd.read_csv(root / DATASET)
    raise FileNotFoundError(DATASET)

class MultiColumnBinarizer(BaseEstimator, TransformerMixin):
    # BaseEstimator: provides parameter handling for sklearn Pipelines and GridSearch
    # TransformerMixin: automatically implements the fit_transform() method

    def __init__(self, min_frequency):
        self.min_frequency = min_frequency
        self.binarizers_ = {}
        self.col_names_ = []
        self.frequent_classes_ = {}


    def _parse_series(self, series):
        series = series.fillna("").astype(str) # Missing values become 0
        parsed = []
        for row in series:
            if not row.strip():
                parsed.append([])
            else:
                items = [
                    item.strip().lower()
                    for item in row.split(",")
                    if item.strip()
                ]
                parsed.append(list(set(items)))
        return parsed

    def _map_rare_to_other(self, lists, freq_set):
        # Maps infrequent or unseen categories into 'other'
        return [list({item if item in freq_set else "other" for item in sublist}) for sublist in lists]

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        self.col_names_ = list(X_df.columns)
        self.binarizers_ = {}
        self.frequent_classes_ = {}

        for col in self.col_names_:
            lists = self._parse_series(X_df[col])
            
            # Count occurrences in the training set and keep classes >= min_frequency
            all_items = [item for sublist in lists for item in sublist]
            counts = pd.Series(all_items).value_counts()
            freq_set = set(counts[counts >= self.min_frequency].index)
            self.frequent_classes_[col] = freq_set

            # Map rare classes to 'other' before fitting MultiLabelBinarizer
            mapped_lists = self._map_rare_to_other(lists, freq_set)
            all_classes = sorted(list(freq_set | {"other"}))

            mlb = MultiLabelBinarizer(classes=all_classes, sparse_output=False)
            mlb.fit(mapped_lists)
            self.binarizers_[col] = mlb

        return self
   

    def transform(self, X):
        X_df = pd.DataFrame(X, columns=self.col_names_)
        transformed_parts = []
        
        for col in self.col_names_:
            lists = self._parse_series(X_df[col])
            mapped_lists = self._map_rare_to_other(lists, self.frequent_classes_[col])
            transformed_parts.append(self.binarizers_[col].transform(mapped_lists))
            
        return np.hstack(transformed_parts)

    def get_feature_names_out(self, input_features=None):
        feature_names = []
        for col in self.col_names_:
            prefix = col.lower().replace(" ", "_")
            classes = self.binarizers_[col].classes_
            feature_names.extend([f"{prefix}_{c.replace(' ', '_')}" for c in classes])
        return np.array(feature_names)

def smotenc_resample(X, y, fattore, random_state):
    if fattore == 1:
        return X, y
    prefixes = tuple(col.lower().replace(" ", "_") + "_" for col in linear_multi_categorical_cols)
    cat_idx = [i for i, col in enumerate(X.columns)
               if col in linear_categorical_cols or col in linear_ordinal_cols or col.startswith(prefixes)]
    return SMOTENC(
        categorical_features=cat_idx,
        sampling_strategy={label: round(n * fattore) for label, n in Counter(y).items()},
        k_neighbors=5,
        random_state=random_state,
    ).fit_resample(X, y)


In [ ]:
# LinearSVC | set C
raw_rbf = load_dataset()

linear_continuous_cols = [
    "Peso (kg)",
    "Altezza (cm)",
    "Neutrofili",
    "Linfociti",
    "Monociti",
    "Eosinofili",
    "Basofili",
    "IgE totali",
]

linear_discrete_cols = [
    "Età",
    "Num Comorbidità",
    "Numero di pregressi interventi ESS",
    "N. allergeni positivi",
    "Giorni al mese di utilizzo spray",
    "ACT",
    "NPS",
    "Nasal Symptoms",
    "Otologic Symptoms",
    "Sleep Symptoms",
    "Emotional Symptoms",
    "Ostruzione nasale",
    "Rinorrea anteriore",
    "Rinorrea posteriore",
    "Starnuti",
    "QoL",
]

linear_categorical_cols = [
    "Categoria Professione",
    "Sesso",
    "Biologico precedente (0 no, 1 sì)",
    "Familiarità Atopia",
    "OCS nell'ultimo anno",
    "Intolleranza ai FANS",
]

linear_multi_categorical_cols = [
    "Comorbidità",
    "Quali allergeni positivi",
]

linear_ordinal_cols = [
    "Fumo Baseline (0=no, 1=ex, 2=si)",
]

linear_numeric = linear_continuous_cols + linear_discrete_cols + linear_ordinal_cols
linear_features = linear_numeric + linear_categorical_cols + linear_multi_categorical_cols
X_linear = raw_rbf.loc[:, linear_features].copy()
y_linear = raw_rbf["TARGET"].astype(int)

for col in linear_numeric:
    values = X_linear[col].astype("string").str.strip()
    european = values.str.contains(",", regex=False, na=False)
    values = values.mask(european, values.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
    X_linear[col] = pd.to_numeric(values, errors="coerce").astype(float)

linear_pre = ColumnTransformer([
    ("num_cont", make_pipeline(SimpleImputer(strategy="mean"), StandardScaler()), linear_continuous_cols),
    ("num_discr", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), linear_discrete_cols),
    ("cat", SimpleImputer(strategy="most_frequent"), linear_categorical_cols),
    ("multi_cat", MultiColumnBinarizer(min_frequency=2), linear_multi_categorical_cols),
    ("ord", make_pipeline(SimpleImputer(strategy="most_frequent"), StandardScaler()), linear_ordinal_cols),
], verbose_feature_names_out=False).set_output(transform="pandas")

linear_post = ColumnTransformer([
    ("cat", OneHotEncoder(sparse_output=False, drop="if_binary", handle_unknown="ignore"), linear_categorical_cols),
], remainder="passthrough")

# Configurazione LinearSVC da Final_Choices.
linear_model = LinearSVC(
    C=0.001,
    loss="squared_hinge",
    dual=False,
    tol=1e-3,
    max_iter=100_000,
    class_weight="balanced",
)
linear_smotenc = FunctionSampler(
    func=smotenc_resample,
    validate=False,
    kw_args={"fattore": 1.8, "random_state": 0},
)
linear_pipeline = ImbPipeline([
    ("pre", linear_pre),
    ("smotenc", linear_smotenc),
    ("post", linear_post),
    ("model", linear_model),
])
linear_pipeline.fit(X_linear, y_linear)
